In [ ]:
# Example 1: Basic GraphQL Schema Definition and Execution
from graphql import (
    GraphQLSchema, GraphQLObjectType, GraphQLField, GraphQLString, 
    GraphQLInt, GraphQLList, GraphQLNonNull, GraphQLID, GraphQLBoolean,
    graphql_sync, build_schema
)

print("Basic GraphQL Schema Definition and Execution:")

# Define sample data
users_data = [
    {"id": "1", "name": "Alice", "email": "alice@example.com", "age": 30, "active": True},
    {"id": "2", "name": "Bob", "email": "bob@example.com", "age": 25, "active": False},
    {"id": "3", "name": "Charlie", "email": "charlie@example.com", "age": 35, "active": True}
]

posts_data = [
    {"id": "1", "title": "GraphQL Basics", "content": "Learning GraphQL", "author_id": "1"},
    {"id": "2", "title": "Python Tips", "content": "Advanced Python", "author_id": "2"},
    {"id": "3", "title": "API Design", "content": "RESTful vs GraphQL", "author_id": "1"}
]

# Resolver functions
def get_user(obj, info, id):
    return next((user for user in users_data if user["id"] == id), None)

def get_all_users(obj, info):
    return users_data

def get_post(obj, info, id):
    post = next((post for post in posts_data if post["id"] == id), None)
    if post:
        author = next((user for user in users_data if user["id"] == post["author_id"]), None)
        return {**post, "author": author}
    return None

def get_all_posts(obj, info):
    posts = []
    for post in posts_data:
        author = next((user for user in users_data if user["id"] == post["author_id"]), None)
        posts.append({**post, "author": author})
    return posts

# Define User type
user_type = GraphQLObjectType(
    name="User",
    fields={
        "id": GraphQLField(GraphQLNonNull(GraphQLID)),
        "name": GraphQLField(GraphQLNonNull(GraphQLString)),
        "email": GraphQLField(GraphQLNonNull(GraphQLString)),
        "age": GraphQLField(GraphQLInt),
        "active": GraphQLField(GraphQLBoolean)
    }
)

# Define Post type with author relationship
post_type = GraphQLObjectType(
    name="Post",
    fields={
        "id": GraphQLField(GraphQLNonNull(GraphQLID)),
        "title": GraphQLField(GraphQLNonNull(GraphQLString)),
        "content": GraphQLField(GraphQLString),
        "author": GraphQLField(user_type)
    }
)

# Define Query type
query_type = GraphQLObjectType(
    name="Query",
    fields={
        "user": GraphQLField(
            user_type,
            args={"id": GraphQLNonNull(GraphQLID)},
            resolve=get_user
        ),
        "users": GraphQLField(
            GraphQLList(user_type),
            resolve=get_all_users
        ),
        "post": GraphQLField(
            post_type,
            args={"id": GraphQLNonNull(GraphQLID)},
            resolve=get_post
        ),
        "posts": GraphQLField(
            GraphQLList(post_type),
            resolve=get_all_posts
        )
    }
)

# Create schema
schema = GraphQLSchema(query=query_type)

# Test queries
print("Testing basic GraphQL queries:")

# Query all users
query1 = "{ users { id name email active } }"
result = graphql_sync(schema, query1)
print(f"All users query result: {result.data}")
if result.errors:
    print(f"Errors: {result.errors}")

# Query specific user
query2 = '{ user(id: "1") { name email age } }'
result = graphql_sync(schema, query2)
print(f"Specific user query result: {result.data}")

# Query posts with author information
query3 = "{ posts { id title author { name email } } }"
result = graphql_sync(schema, query3)
print(f"Posts with authors query result: {result.data}")

# Test error handling with invalid ID
query4 = '{ user(id: "999") { name } }'
result = graphql_sync(schema, query4)
print(f"Invalid user query result: {result.data}")

print("✅ Basic GraphQL schema definition and execution completed")

## Example 2: Schema Definition Language (SDL) and Build Schema

In [ ]:
# Example 2: Schema Definition Language (SDL) and Build Schema
from graphql import build_schema, graphql_sync, GraphQLError
from graphql.utilities import print_schema

print("Schema Definition Language (SDL) and Build Schema:")

# Define schema using SDL (Schema Definition Language)
sdl_schema = """
    type User {
        id: ID!
        username: String!
        email: String!
        profile: UserProfile
        posts: [Post!]!
        createdAt: String!
    }
    
    type UserProfile {
        firstName: String
        lastName: String
        bio: String
        avatar: String
        website: String
    }
    
    type Post {
        id: ID!
        title: String!
        content: String!
        status: PostStatus!
        author: User!
        tags: [String!]!
        createdAt: String!
        updatedAt: String!
    }
    
    enum PostStatus {
        DRAFT
        PUBLISHED
        ARCHIVED
    }
    
    input CreateUserInput {
        username: String!
        email: String!
        firstName: String
        lastName: String
    }
    
    input UpdatePostInput {
        title: String
        content: String
        status: PostStatus
        tags: [String!]
    }
    
    type Query {
        user(id: ID!): User
        users(limit: Int = 10): [User!]!
        post(id: ID!): Post
        posts(status: PostStatus, authorId: ID): [Post!]!
        searchPosts(query: String!): [Post!]!
    }
    
    type Mutation {
        createUser(input: CreateUserInput!): User!
        updatePost(id: ID!, input: UpdatePostInput!): Post
        deletePost(id: ID!): Boolean!
    }
"""

# Extended sample data for SDL example
sdl_users_data = [
    {
        "id": "1",
        "username": "alice_dev",
        "email": "alice@example.com",
        "profile": {
            "firstName": "Alice",
            "lastName": "Johnson",
            "bio": "Full-stack developer",
            "avatar": "https://example.com/alice.jpg",
            "website": "https://alice.dev"
        },
        "createdAt": "2023-01-15T10:00:00Z"
    },
    {
        "id": "2",
        "username": "bob_writer",
        "email": "bob@example.com",
        "profile": {
            "firstName": "Bob",
            "lastName": "Smith",
            "bio": "Technical writer",
            "avatar": "https://example.com/bob.jpg",
            "website": ""
        },
        "createdAt": "2023-02-20T14:30:00Z"
    }
]

sdl_posts_data = [
    {
        "id": "1",
        "title": "GraphQL Core Fundamentals",
        "content": "Deep dive into GraphQL core concepts and implementation details.",
        "status": "PUBLISHED",
        "author_id": "1",
        "tags": ["graphql", "python", "tutorial"],
        "createdAt": "2023-09-01T09:00:00Z",
        "updatedAt": "2023-09-05T11:30:00Z"
    },
    {
        "id": "2",
        "title": "Advanced GraphQL Techniques",
        "content": "Exploring advanced GraphQL patterns and best practices.",
        "status": "DRAFT",
        "author_id": "1",
        "tags": ["graphql", "advanced", "patterns"],
        "createdAt": "2023-09-10T16:00:00Z",
        "updatedAt": "2023-09-15T10:15:00Z"
    },
    {
        "id": "3",
        "title": "Writing Technical Documentation",
        "content": "Best practices for writing clear and effective technical documentation.",
        "status": "PUBLISHED",
        "author_id": "2",
        "tags": ["documentation", "writing", "technical"],
        "createdAt": "2023-09-12T13:45:00Z",
        "updatedAt": "2023-09-12T13:45:00Z"
    }
]

# Resolver functions for SDL schema
def resolve_user(obj, info, id):
    user = next((u for u in sdl_users_data if u["id"] == id), None)
    if user:
        # Add posts to user
        user_posts = [p for p in sdl_posts_data if p["author_id"] == id]
        user["posts"] = user_posts
    return user

def resolve_users(obj, info, limit=10):
    limited_users = sdl_users_data[:limit]
    for user in limited_users:
        user_posts = [p for p in sdl_posts_data if p["author_id"] == user["id"]]
        user["posts"] = user_posts
    return limited_users

def resolve_post(obj, info, id):
    post = next((p for p in sdl_posts_data if p["id"] == id), None)
    if post:
        author = next((u for u in sdl_users_data if u["id"] == post["author_id"]), None)
        post["author"] = author
    return post

def resolve_posts(obj, info, status=None, authorId=None):
    posts = sdl_posts_data
    
    if status:
        posts = [p for p in posts if p["status"] == status]
    
    if authorId:
        posts = [p for p in posts if p["author_id"] == authorId]
    
    # Add author information
    for post in posts:
        author = next((u for u in sdl_users_data if u["id"] == post["author_id"]), None)
        post["author"] = author
    
    return posts

def resolve_search_posts(obj, info, query):
    search_term = query.lower()
    matching_posts = []
    
    for post in sdl_posts_data:
        if (search_term in post["title"].lower() or 
            search_term in post["content"].lower() or
            any(search_term in tag.lower() for tag in post["tags"])):
            
            author = next((u for u in sdl_users_data if u["id"] == post["author_id"]), None)
            post_copy = dict(post)
            post_copy["author"] = author
            matching_posts.append(post_copy)
    
    return matching_posts

def resolve_create_user(obj, info, input):
    new_id = str(len(sdl_users_data) + 1)
    new_user = {
        "id": new_id,
        "username": input["username"],
        "email": input["email"],
        "profile": {
            "firstName": input.get("firstName", ""),
            "lastName": input.get("lastName", ""),
            "bio": "",
            "avatar": "",
            "website": ""
        },
        "createdAt": "2023-09-15T12:00:00Z",
        "posts": []
    }
    sdl_users_data.append(new_user)
    return new_user

def resolve_update_post(obj, info, id, input):
    post_index = next((i for i, p in enumerate(sdl_posts_data) if p["id"] == id), None)
    if post_index is None:
        return None
    
    # Update post fields
    for key, value in input.items():
        sdl_posts_data[post_index][key] = value
    
    sdl_posts_data[post_index]["updatedAt"] = "2023-09-15T12:30:00Z"
    
    # Add author information
    post = sdl_posts_data[post_index]
    author = next((u for u in sdl_users_data if u["id"] == post["author_id"]), None)
    post["author"] = author
    
    return post

def resolve_delete_post(obj, info, id):
    post_index = next((i for i, p in enumerate(sdl_posts_data) if p["id"] == id), None)
    if post_index is not None:
        sdl_posts_data.pop(post_index)
        return True
    return False

# Build schema from SDL
sdl_schema_obj = build_schema(sdl_schema)

# Set resolvers. graphql-core 3 reads a field's resolver from its `.resolve` attribute.
sdl_schema_obj.query_type.fields["user"].resolve = resolve_user
sdl_schema_obj.query_type.fields["users"].resolve = resolve_users
sdl_schema_obj.query_type.fields["post"].resolve = resolve_post
sdl_schema_obj.query_type.fields["posts"].resolve = resolve_posts
sdl_schema_obj.query_type.fields["searchPosts"].resolve = resolve_search_posts

sdl_schema_obj.mutation_type.fields["createUser"].resolve = resolve_create_user
sdl_schema_obj.mutation_type.fields["updatePost"].resolve = resolve_update_post
sdl_schema_obj.mutation_type.fields["deletePost"].resolve = resolve_delete_post

print("Testing SDL-built schema:")

# Test complex query with nested fields
complex_query = """
{
  users(limit: 2) {
    id
    username
    profile {
      firstName
      lastName
      bio
    }
    posts {
      title
      status
      tags
    }
  }
}
"""

result = graphql_sync(sdl_schema_obj, complex_query)
assert not result.errors and result.data["users"], result
print(f"Complex nested query result: {result.data}")

# Test filtering query
filter_query = """
{
  posts(status: PUBLISHED) {
    id
    title
    author {
      username
      profile {
        firstName
      }
    }
    tags
  }
}
"""

result = graphql_sync(sdl_schema_obj, filter_query)
assert not result.errors and result.data["posts"], result
print(f"Filtered posts query result: {result.data}")

# Test search functionality
search_query = """
{
  searchPosts(query: "graphql") {
    id
    title
    tags
    author {
      username
    }
  }
}
"""

result = graphql_sync(sdl_schema_obj, search_query)
assert not result.errors and result.data["searchPosts"], result
print(f"Search posts query result: {result.data}")

# Test mutation
mutation_query = """
mutation {
  createUser(input: {
    username: "charlie_designer"
    email: "charlie@example.com"
    firstName: "Charlie"
    lastName: "Brown"
  }) {
    id
    username
    profile {
      firstName
      lastName
    }
  }
}
"""

result = graphql_sync(sdl_schema_obj, mutation_query)
assert not result.errors and result.data["createUser"], result
print(f"Create user mutation result: {result.data}")

# Print schema definition
print(f"\nSchema SDL representation (first 200 chars):")
schema_sdl = print_schema(sdl_schema_obj)
print(schema_sdl[:200] + "...")

print("✅ Schema Definition Language (SDL) and build schema completed")

## Example 3: Query Parsing, Validation, and AST Analysis

In [ ]:
# Example 3: Query Parsing, Validation, and AST Analysis
from graphql import (
    parse, validate, visit, GraphQLError,
    ValidationRule, NoUnusedFragmentsRule
)
from graphql.language import Node, Visitor, print_ast

print("Query Parsing, Validation, and AST Analysis:")

# Sample queries for parsing and validation
valid_query = """
query GetUserProfile($userId: ID!, $includePosts: Boolean = false) {
  user(id: $userId) {
    id
    username
    email
    profile {
      firstName
      lastName
      bio
      website
    }
    posts @include(if: $includePosts) {
      id
      title
      status
      createdAt
    }
    ...UserDetails
  }
}

fragment UserDetails on User {
  id
  username
  profile {
    firstName
    lastName
  }
}
"""

invalid_query = """
query GetUser {
  user(id: "1") {
    id
    nonExistentField
    unknownRelation {
      someField
    }
  }
}
"""

complex_query = """
query ComplexQuery($status: PostStatus) {
  posts(status: $status) {
    id
    title
    author {
      ...UserInfo
    }
    ...PostDetails
  }
}

fragment UserInfo on User {
  id
  username
  profile {
    firstName
    lastName
  }
}

fragment PostDetails on Post {
  content
  tags
  createdAt
  updatedAt
}
"""

# Parse queries into AST
print("1. Query Parsing:")
try:
    valid_ast = parse(valid_query)
    print(f"✅ Valid query parsed successfully")
    print(f"AST type: {type(valid_ast)}")
    print(f"Definitions count: {len(valid_ast.definitions)}")
    
    # Print parsed AST (first 200 chars)
    ast_string = print_ast(valid_ast)
    print(f"AST representation (first 200 chars): {ast_string[:200]}...")
    
except Exception as e:
    print(f"❌ Error parsing valid query: {e}")
    raise

try:
    complex_ast = parse(complex_query)
    print(f"✅ Complex query with fragments parsed successfully")
    print(f"Definitions count: {len(complex_ast.definitions)}")
    
    # Analyze definitions
    for i, definition in enumerate(complex_ast.definitions):
        def_name = getattr(definition, 'name', None)
        name_value = def_name.value if def_name else 'unnamed'
        print(f"  Definition {i}: {definition.kind} - {name_value}")
        
except Exception as e:
    print(f"❌ Error parsing complex query: {e}")
    raise

# Query validation
print(f"\n2. Query Validation:")

# Validate valid query. It must USE the fragment it defines: an unused fragment makes the document invalid under the
# spec's NoUnusedFragments rule (measured: without the `...UserDetails` spread, validate() returns
# "Fragment 'UserDetails' is never used.")
valid_errors = validate(sdl_schema_obj, valid_ast)
assert not valid_errors, [error.message for error in valid_errors]
print(f"✅ Valid query passed validation")

# Validate invalid query: it parses (the problems are semantic), and validate() must name both unknown fields
invalid_ast = parse(invalid_query)
invalid_errors = validate(sdl_schema_obj, invalid_ast)
invalid_messages = [error.message for error in invalid_errors]
assert any("nonExistentField" in m for m in invalid_messages), invalid_messages
assert any("unknownRelation" in m for m in invalid_messages), invalid_messages
print(f"✅ Invalid query correctly caught validation errors:")
for message in invalid_messages[:3]:  # Show first 3
    print(f"  - {message}")

# Custom validation rules using factory function
print(f"\n3. Custom Validation Rules:")

def max_depth_rule_factory(max_depth=5):
    """Factory function to create max depth validation rule"""
    class MaxDepthRule(ValidationRule):
        def __init__(self, context):
            super().__init__(context)
            self.max_depth = max_depth
            self.current_depth = 0

        def enter_field(self, node, *_):
            self.current_depth += 1
            if self.current_depth > self.max_depth:
                self.report_error(
                    GraphQLError(
                        f"Query depth {self.current_depth} exceeds maximum allowed depth {self.max_depth}",
                        node
                    )
                )

        def leave_field(self, node, *_):
            self.current_depth -= 1
    
    return MaxDepthRule

# Apply custom validation rules
custom_rules = [max_depth_rule_factory(max_depth=3), NoUnusedFragmentsRule]
custom_errors = validate(sdl_schema_obj, complex_ast, custom_rules)

# complex_query nests fields 2 deep and uses both fragments it defines (measured: no errors)
assert not custom_errors, [error.message for error in custom_errors]
print(f"✅ Complex query passed custom validation")

# AST Analysis using manual tree traversal
print(f"\n4. AST Analysis with Manual Traversal:")

def analyze_ast(node, stats=None, depth=0):
    """Recursively analyze AST (depth = how many fields deep)"""
    if stats is None:
        stats = {
            'field_count': 0,
            'fragment_count': 0,
            'variable_count': 0,
            'directive_count': 0,
            'max_depth': 0,
            'fields_accessed': [],
            'fragments_defined': []
        }
    
    # Get node kind
    kind = getattr(node, 'kind', None)
    
    if kind == 'field':
        depth += 1
        stats['max_depth'] = max(stats['max_depth'], depth)
        stats['field_count'] += 1
        if hasattr(node, 'name') and node.name:
            stats['fields_accessed'].append(node.name.value)
    elif kind == 'fragment_definition':
        stats['fragment_count'] += 1
        if hasattr(node, 'name') and node.name:
            stats['fragments_defined'].append(node.name.value)
    elif kind == 'variable_definition':
        stats['variable_count'] += 1
    elif kind == 'directive':
        stats['directive_count'] += 1
    
    # Recursively traverse children. graphql-core stores a node's fields in __slots__ listed in `node.keys`;
    # child lists are tuples.
    for key in node.keys:
        value = getattr(node, key)
        if isinstance(value, (list, tuple)):
            for item in value:
                if isinstance(item, Node):
                    analyze_ast(item, stats, depth)
        elif isinstance(value, Node):
            analyze_ast(value, stats, depth)
    
    return stats

# Analyze valid query
query_stats = analyze_ast(valid_ast)

print(f"Query Analysis Results:")
print(f"  Fields accessed: {query_stats['field_count']}")
unique_fields = sorted(set(query_stats['fields_accessed']))
print(f"  Unique fields: {unique_fields[:5]}...")
print(f"  Fragments defined: {query_stats['fragment_count']} ({query_stats['fragments_defined']})")
print(f"  Variables: {query_stats['variable_count']}")
print(f"  Directives: {query_stats['directive_count']}")
print(f"  Max depth: {query_stats['max_depth']}")
# Measured on valid_query: 14 fields in the operation + 5 in UserDetails (13 distinct names), $userId and $includePosts,
# one @include, and user > profile/posts > their subfields is 3 fields deep
assert (query_stats['field_count'], len(unique_fields)) == (19, 13), query_stats
assert query_stats['fragments_defined'] == ['UserDetails'], query_stats
assert (query_stats['variable_count'], query_stats['directive_count'], query_stats['max_depth']) == (2, 1, 3), query_stats

# Query complexity analysis
print(f"\n5. Query Complexity Analysis:")

class ComplexityVisitor(Visitor):
    """Query complexity with graphql's own visit(): 1 per field, 2 per fragment spread, 1 per inline fragment"""
    def __init__(self):
        super().__init__()
        self.score = 0
        self.depth = 0
        self.max_depth = 0

    def enter_field(self, node, *_):
        self.score += 1
        self.depth += 1
        self.max_depth = max(self.max_depth, self.depth)

    def leave_field(self, node, *_):
        self.depth -= 1

    def enter_fragment_spread(self, node, *_):
        self.score += 2  # Fragments add complexity

    def enter_inline_fragment(self, node, *_):
        self.score += 1

# Analyze complexity
complexity_visitor = ComplexityVisitor()
visit(complex_ast, complexity_visitor)
complexity_score, max_depth = complexity_visitor.score, complexity_visitor.max_depth

print(f"Complexity Analysis:")
print(f"  Complexity Score: {complexity_score}")
print(f"  Maximum Depth: {max_depth}")
# Measured: 13 fields (4 in the operation, 5 in UserInfo, 4 in PostDetails) + 2 spreads x 2 = 17; fields nest 2 deep
assert (complexity_score, max_depth) == (17, 2), (complexity_score, max_depth)

# Introspection query
print(f"\n6. Introspection Query:")

introspection_query = """
query IntrospectionQuery {
  __schema {
    types {
      name
      kind
    }
  }
}
"""

try:
    introspection_ast = parse(introspection_query)
    introspection_errors = validate(sdl_schema_obj, introspection_ast)
    
    if introspection_errors:
        print(f"❌ Introspection query validation errors:")
        for error in introspection_errors:
            print(f"  - {error.message}")
    else:
        print(f"✅ Introspection query is valid")

    # Execute introspection
    introspection_result = graphql_sync(sdl_schema_obj, introspection_query)
    if introspection_result.data:
        schema_types = introspection_result.data["__schema"]["types"]
        type_names = [t["name"] for t in schema_types if not t["name"].startswith("__")]
        print(f"Schema contains {len(type_names)} custom types: {type_names[:5]}...")

except Exception as e:
    print(f"Error with introspection: {e}")
    raise

print("✅ Query parsing, validation, and AST analysis completed")

## Example 4: Advanced Execution Features and Middleware

In [ ]:
# Example 4: Advanced Execution Features and Middleware
from graphql import (
    graphql_sync, ExecutionResult, GraphQLError, 
    MiddlewareManager, Middleware
)
import time
import asyncio

print("Advanced Execution Features and Middleware:")

# Custom middleware classes
class TimingMiddleware:
    """Middleware to track execution time for each field"""
    
    def resolve(self, next, root, info, **args):
        start_time = time.time()
        field_name = info.field_name
        path = ".".join(str(p) for p in info.path.as_list())
        
        try:
            result = next(root, info, **args)
            execution_time = time.time() - start_time
            print(f"⏱️ Field '{path}' executed in {execution_time:.3f}s")
            return result
        except Exception as e:
            execution_time = time.time() - start_time
            print(f"❌ Field '{path}' failed in {execution_time:.3f}s: {e}")
            raise

class CachingMiddleware:
    """Simple caching middleware for expensive operations"""
    
    def __init__(self):
        self.cache = {}
    
    def resolve(self, next, root, info, **args):
        # Create cache key from field name and arguments
        cache_key = f"{info.field_name}:{str(sorted(args.items()))}"
        
        # Check cache first
        if cache_key in self.cache:
            print(f"💾 Cache hit for {info.field_name}")
            return self.cache[cache_key]
        
        # Execute resolver and cache result
        result = next(root, info, **args)
        
        # Only cache non-null results
        if result is not None:
            self.cache[cache_key] = result
            print(f"🔄 Cached result for {info.field_name}")
        
        return result

class LoggingMiddleware:
    """Comprehensive logging middleware"""
    
    def __init__(self):
        self.failures = []  # "Type.field" of every resolver error that passed through (asserted below)
    
    def resolve(self, next, root, info, **args):
        field_name = info.field_name
        parent_type = info.parent_type.name if info.parent_type else "Root"
        
        print(f"🔍 Executing {parent_type}.{field_name}")
        
        # Log arguments if present
        if args:
            print(f"   Args: {args}")
        
        # Log context information
        context = info.context or {}
        if context.get('user_id'):
            print(f"   User: {context['user_id']}")
        
        try:
            result = next(root, info, **args)
            print(f"✅ {parent_type}.{field_name} completed")
            return result
        except Exception as e:
            # log WHICH field raised WHAT and pass it on; the message is the executor's to report (it lands in
            # result.errors, which both queries that run this middleware assert)
            self.failures.append(f"{parent_type}.{field_name}")
            print(f"↩️ {parent_type}.{field_name} raised {type(e).__name__}; passed on to the executor")
            raise

class AuthorizationMiddleware:
    """Authorization middleware for field-level access control"""
    
    def __init__(self):
        # Define field-level permissions
        self.permissions = {
            "User.email": ["admin", "owner"],
            "Post.content": ["user", "admin", "owner"],
            "createUser": ["admin"],
            "deletePost": ["admin", "owner"]
        }
    
    def resolve(self, next, root, info, **args):
        field_name = info.field_name
        parent_type = info.parent_type.name if info.parent_type else "Root"
        field_key = f"{parent_type}.{field_name}"
        
        # Check if field requires permissions
        required_roles = self.permissions.get(field_key, [])
        
        if required_roles:
            context = info.context or {}
            user_role = context.get('user_role', 'guest')
            
            # Check authorization
            if user_role not in required_roles and 'admin' not in [user_role]:
                raise GraphQLError(
                    f"Access denied to field '{field_key}'. Required role: {required_roles}",
                    info.field_nodes
                )
            
            print(f"🔐 Authorized access to {field_key} for role: {user_role}")
        
        return next(root, info, **args)

# Custom execution context
class ExecutionContext:
    def __init__(self, user_id=None, user_role="guest", request_id=None):
        self.user_id = user_id
        self.user_role = user_role
        self.request_id = request_id or f"req_{int(time.time())}"
        self.start_time = time.time()
    
    def to_dict(self):
        return {
            "user_id": self.user_id,
            "user_role": self.user_role,
            "request_id": self.request_id,
            "start_time": self.start_time
        }

# Test middleware with different execution contexts
print("1. Testing Middleware Stack:")

# Create middleware instances
timing_middleware = TimingMiddleware()
caching_middleware = CachingMiddleware()
logging_middleware = LoggingMiddleware()
auth_middleware = AuthorizationMiddleware()

# Test with admin context
admin_context = ExecutionContext(user_id="admin_1", user_role="admin").to_dict()

admin_query = """
{
  users(limit: 2) {
    id
    username
    email
    profile {
      firstName
      lastName
    }
  }
}
"""

print("\n📋 Admin Query Execution:")
result = graphql_sync(
    sdl_schema_obj, 
    admin_query, 
    context_value=admin_context,
    middleware=[timing_middleware, caching_middleware, logging_middleware, auth_middleware]
)

assert not result.errors, f"admin query failed: {result.errors}"
print(f"✅ Admin query completed successfully")

# Test with regular user context
user_context = ExecutionContext(user_id="user_1", user_role="user").to_dict()

user_query = """
{
  posts {
    id
    title
    content
    author {
      username
      email
    }
  }
}
"""

print("\n👤 User Query Execution:")
result = graphql_sync(
    sdl_schema_obj, 
    user_query, 
    context_value=user_context,
    middleware=[timing_middleware, auth_middleware]
)

# role 'user' may read Post.content but not User.email, so the query must fail on that field and only on it
denied = [str(e) for e in (result.errors or [])]
assert denied and all("Access denied to field 'User.email'" in d for d in denied), denied
print(f"Expected authorization errors: {denied}")

# Test caching middleware
print("\n2. Testing Caching Middleware:")

cached_query = """
{
  user(id: "1") {
    username
    profile {
      firstName
      lastName
    }
  }
}
"""

# First execution (no cache)
print("First execution:")
result1 = graphql_sync(
    sdl_schema_obj,
    cached_query,
    middleware=[caching_middleware, timing_middleware]
)

# Second execution (should use cache)
print("\nSecond execution:")
result2 = graphql_sync(
    sdl_schema_obj,
    cached_query,
    middleware=[caching_middleware, timing_middleware]
)

# Custom error handling middleware
class ErrorHandlingMiddleware:
    """Middleware for custom error handling and formatting"""
    
    def resolve(self, next, root, info, **args):
        try:
            return next(root, info, **args)
        except ValueError as e:
            # Convert ValueError to GraphQLError with custom message
            raise GraphQLError(
                f"Validation error in {info.field_name}: {str(e)}",
                info.field_nodes,
                extensions={"code": "VALIDATION_ERROR", "field": info.field_name}
            )
        except KeyError as e:
            # Handle missing data gracefully
            raise GraphQLError(
                f"Data not found for {info.field_name}: {str(e)}",
                info.field_nodes,
                extensions={"code": "NOT_FOUND", "field": info.field_name}
            )
        except Exception as e:
            # Generic error handling
            raise GraphQLError(
                f"Internal error in {info.field_name}: {str(e)}",
                info.field_nodes,
                extensions={"code": "INTERNAL_ERROR", "field": info.field_name}
            )

# Test error handling
print("\n3. Testing Error Handling Middleware:")

# Create a resolver that throws an error
def error_resolver(obj, info):
    raise ValueError("This is a test validation error")

# Temporarily add error field for testing
error_field = GraphQLField(GraphQLString, resolve=error_resolver)
sdl_schema_obj.query_type.fields["testError"] = error_field

error_query = """
{
  testError
}
"""

error_middleware = ErrorHandlingMiddleware()
result = graphql_sync(
    sdl_schema_obj,
    error_query,
    middleware=[error_middleware, logging_middleware]
)

# the ValueError raised by the resolver must come back as the middleware's GraphQLError, with its extensions
assert result.errors and result.errors[0].extensions == {"code": "VALIDATION_ERROR", "field": "testError"}, result.errors
assert [e.message for e in result.errors] == ["Validation error in testError: This is a test validation error"], result.errors
# ...and it is the ONLY resolver error the logging middleware saw in this cell (the admin query above ran through it too)
assert logging_middleware.failures == ["Query.testError"], logging_middleware.failures
for error in result.errors:
    print(f"🚨 {error.path} rejected as expected, code {error.extensions['code']} (message asserted above)")
    if hasattr(error, 'extensions'):
        print(f"   Extensions: {error.extensions}")

# Performance monitoring middleware
class PerformanceMiddleware:
    """Middleware for performance monitoring and reporting"""
    
    def __init__(self):
        self.field_metrics = {}
    
    def resolve(self, next, root, info, **args):
        field_name = info.field_name
        start_time = time.time()
        
        try:
            result = next(root, info, **args)
            execution_time = time.time() - start_time
            
            # Record metrics
            if field_name not in self.field_metrics:
                self.field_metrics[field_name] = {
                    "count": 0,
                    "total_time": 0,
                    "avg_time": 0,
                    "max_time": 0
                }
            
            metrics = self.field_metrics[field_name]
            metrics["count"] += 1
            metrics["total_time"] += execution_time
            metrics["avg_time"] = metrics["total_time"] / metrics["count"]
            metrics["max_time"] = max(metrics["max_time"], execution_time)
            
            return result
            
        except Exception as e:
            execution_time = time.time() - start_time
            print(f"⚠️ Field {field_name} failed after {execution_time:.3f}s")
            raise
    
    def get_metrics_report(self):
        print("\n📊 Performance Metrics Report:")
        for field_name, metrics in self.field_metrics.items():
            print(f"  {field_name}:")
            print(f"    Calls: {metrics['count']}")
            print(f"    Avg Time: {metrics['avg_time']:.3f}s")
            print(f"    Max Time: {metrics['max_time']:.3f}s")
            print(f"    Total Time: {metrics['total_time']:.3f}s")

# Test performance monitoring
print("\n4. Testing Performance Monitoring:")

perf_middleware = PerformanceMiddleware()

# Execute multiple queries
test_queries = [
    '{ users { id username } }',
    '{ posts { id title } }',
    '{ user(id: "1") { username profile { firstName } } }'
]

for i, query in enumerate(test_queries):
    print(f"\nExecuting test query {i + 1}:")
    result = graphql_sync(
        sdl_schema_obj,
        query,
        middleware=[perf_middleware]
    )

# Generate performance report
perf_middleware.get_metrics_report()

print("✅ Advanced execution features and middleware completed")

## Example 5: Production GraphQL Core Integration and Performance Optimization

In [ ]:
# Example 5: Production GraphQL Core Integration and Performance Optimization
from graphql import (
    GraphQLError,
    get_introspection_query, build_client_schema
)
from graphql.utilities import strip_ignored_characters
import json
import concurrent.futures
import threading
import time

print("Production GraphQL Core Integration and Performance Optimization:")

# Production-ready execution engine
class ProductionGraphQLEngine:
    def __init__(self, schema, middleware=None):
        self.schema = schema
        self.middleware = middleware or []
        self.query_cache = {}
        self.execution_stats = {
            "total_queries": 0,
            "cache_hits": 0,
            "errors": 0,
            "avg_execution_time": 0,
            "total_execution_time": 0
        }
        self.lock = threading.Lock()
    
    def execute_query(self, query, variables=None, context=None, operation_name=None):
        """Execute GraphQL query with caching and statistics"""
        start_time = time.time()
        
        with self.lock:
            self.execution_stats["total_queries"] += 1
        
        try:
            # Normalize query for caching
            normalized_query = strip_ignored_characters(query)
            cache_key = f"{normalized_query}:{json.dumps(variables, sort_keys=True) if variables else 'null'}"
            
            # Check query cache (for queries without variables or with static variables)
            if not variables or self._is_cacheable_query(query):
                if cache_key in self.query_cache:
                    with self.lock:
                        self.execution_stats["cache_hits"] += 1
                    print(f"🚀 Cache hit for query")
                    return self.query_cache[cache_key]
            
            # Execute query
            result = graphql_sync(
                self.schema,
                query,
                variable_values=variables,
                context_value=context,
                operation_name=operation_name,
                middleware=self.middleware
            )
            
            # Cache successful queries
            if not result.errors and self._is_cacheable_query(query):
                self.query_cache[cache_key] = result
            
            # Update statistics
            execution_time = time.time() - start_time
            with self.lock:
                if result.errors:
                    self.execution_stats["errors"] += 1
                
                self.execution_stats["total_execution_time"] += execution_time
                self.execution_stats["avg_execution_time"] = (
                    self.execution_stats["total_execution_time"] / 
                    self.execution_stats["total_queries"]
                )
            
            return result
            
        except Exception as e:
            execution_time = time.time() - start_time
            with self.lock:
                self.execution_stats["errors"] += 1
                self.execution_stats["total_execution_time"] += execution_time
                self.execution_stats["avg_execution_time"] = (
                    self.execution_stats["total_execution_time"] / 
                    self.execution_stats["total_queries"]
                )
            
            from graphql import ExecutionResult
            return ExecutionResult(
                data=None,
                errors=[GraphQLError(f"Execution failed: {str(e)}")]
            )
    
    def _is_cacheable_query(self, query):
        """Determine if query is cacheable (simple heuristic)"""
        # Don't cache mutations or queries with dynamic content
        return "mutation" not in query.lower() and "now()" not in query.lower()
    
    def get_stats(self):
        with self.lock:
            return dict(self.execution_stats)
    
    def clear_cache(self):
        with self.lock:
            self.query_cache.clear()
            print("🧹 Query cache cleared")

# Create production engine
prod_engine = ProductionGraphQLEngine(
    sdl_schema_obj,
    middleware=[TimingMiddleware(), LoggingMiddleware()]
)

print("1. Testing Production Execution Engine:")

# Test queries for performance
test_queries = [
    {
        "query": '{ users { id username } }',
        "variables": None,
        "name": "Simple user list"
    },
    {
        # an operation must DECLARE the variables it uses (measured: the anonymous form fails with
        # "Variable '$userId' is not defined")
        "query": 'query GetUser($userId: ID!) { user(id: $userId) { username profile { firstName lastName } } }',
        "variables": {"userId": "1"},
        "name": "Parameterized user query"
    },
    {
        "query": '{ posts(status: PUBLISHED) { id title author { username } } }',
        "variables": None,
        "name": "Filtered posts query"
    }
]

# Execute queries multiple times to test caching
for iteration in range(2):
    print(f"\n--- Iteration {iteration + 1} ---")
    for test in test_queries:
        print(f"\nExecuting: {test['name']}")
        result = prod_engine.execute_query(
            test["query"],
            variables=test["variables"]
        )
        
        assert not result.errors, f"{test['name']}: {[str(e)[:80] for e in result.errors]}"
        print(f"✅ Success")

# Show execution statistics
stats = prod_engine.get_stats()
print(f"\n📈 Execution Statistics:")
for key, value in stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.3f}")
    else:
        print(f"  {key}: {value}")

# Concurrent execution testing
print("\n2. Testing Concurrent Execution:")

def execute_concurrent_query(engine, query_info):
    """Execute a query in a thread"""
    thread_id = threading.current_thread().ident
    start_time = time.time()
    
    result = engine.execute_query(
        query_info["query"],
        variables=query_info.get("variables")
    )
    
    execution_time = time.time() - start_time
    return {
        "thread_id": thread_id,
        "execution_time": execution_time,
        "success": not result.errors,
        "query_name": query_info["name"]
    }

# Execute queries concurrently
concurrent_queries = test_queries * 2  # Run each query 2 times

with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    futures = [
        executor.submit(execute_concurrent_query, prod_engine, query)
        for query in concurrent_queries
    ]
    
    concurrent_results = [future.result() for future in concurrent.futures.as_completed(futures)]

print(f"Concurrent execution completed: {len(concurrent_results)} queries")
success_count = sum(1 for r in concurrent_results if r["success"])
print(f"  Successful: {success_count}/{len(concurrent_results)}")
assert success_count == len(concurrent_results), concurrent_results

# Updated stats after concurrent execution
final_stats = prod_engine.get_stats()
print(f"\n📊 Final Statistics:")
for key, value in final_stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.3f}")
    else:
        print(f"  {key}: {value}")

# Schema introspection and client schema building
print("\n3. Testing Schema Introspection and Client Schema Building:")

# Get introspection query
introspection_query = get_introspection_query()
print(f"Introspection query length: {len(introspection_query)} characters")

# Execute introspection
introspection_result = prod_engine.execute_query(introspection_query)

if introspection_result.errors:
    raise AssertionError(f"introspection failed: {[str(e)[:100] for e in introspection_result.errors]}")
else:
    print(f"✅ Introspection successful")
    
    # Build client schema from introspection result
    try:
        client_schema = build_client_schema(introspection_result.data)
        print(f"✅ Client schema built successfully")
        print(f"Client schema types: {len(client_schema.type_map)} types")
        
        # List some types
        user_types = [name for name in client_schema.type_map.keys() if not name.startswith('__')]
        print(f"Custom types: {user_types[:5]}...")
        
    except Exception as e:
        raise AssertionError(f"build_client_schema failed on our own introspection result: {str(e)[:100]}") from e

# Query optimization and analysis
print("\n4. Query Optimization and Analysis:")

class QueryOptimizer:
    """Analyze and optimize GraphQL queries"""
    
    @staticmethod
    def suggest_optimizations(query):
        """Suggest query optimizations"""
        suggestions = []
        
        if "posts" in query and "author" in query:
            suggestions.append("Consider using fragments for repeated author fields")
        
        if query.count("{") > 5:
            suggestions.append("Query has high nesting - consider breaking into multiple queries")
        
        if "limit" not in query and ("users" in query or "posts" in query):
            suggestions.append("Consider adding limit parameter for list queries")
        
        return suggestions

optimizer = QueryOptimizer()

# Analyze test queries
for test in test_queries:
    query = test["query"]
    print(f"\nAnalyzing: {test['name']}")
    
    suggestions = optimizer.suggest_optimizations(query)
    if suggestions:
        print(f"  Suggestions: {suggestions[0]}")
    else:
        print(f"  ✅ Query looks optimized")

# Error formatting and handling
print("\n5. Production Error Formatting:")

class ProductionErrorFormatter:
    """Format errors for production use"""
    
    @staticmethod
    def format_error_for_client(error):
        """Format GraphQL error for client consumption"""
        formatted = {
            "message": error.message,
            "path": error.path if error.path else None,
        }
        
        if error.locations:
            formatted["locations"] = [
                {"line": loc.line, "column": loc.column}
                for loc in error.locations
            ]
        
        # Add extensions if present
        if hasattr(error, 'extensions') and error.extensions:
            formatted["extensions"] = error.extensions
        
        # Don't expose sensitive information in production
        if "Internal error" in error.message:
            formatted["message"] = "An internal error occurred"
            formatted["extensions"] = {"code": "INTERNAL_ERROR"}
        
        return formatted
    
    @staticmethod
    def format_execution_result(result):
        """Format complete execution result"""
        formatted_result = {"data": result.data}
        
        if result.errors:
            formatted_result["errors"] = [
                ProductionErrorFormatter.format_error_for_client(error)
                for error in result.errors
            ]
        
        return formatted_result

# Test error formatting
error_formatter = ProductionErrorFormatter()

# Create a query that will cause an error
error_query = '{ user(id: "999") { nonExistentField } }'
error_result = prod_engine.execute_query(error_query)

print("Error formatting example:")
if error_result.errors:
    formatted_result = error_formatter.format_execution_result(error_result)
    print(f"  Formatted {len(error_result.errors)} error(s)")
    print(f"  Sample: {formatted_result['errors'][0]['message'][:60]}...")

# Performance benchmarking
print("\n6. Performance Benchmarking:")

def benchmark_queries(engine, queries, iterations=5):
    """Benchmark query performance"""
    results = {}
    
    for query_info in queries:
        query_name = query_info["name"]
        query = query_info["query"]
        variables = query_info.get("variables")
        
        times = []
        for _ in range(iterations):
            start_time = time.time()
            result = engine.execute_query(query, variables=variables)
            execution_time = time.time() - start_time
            times.append(execution_time)
        
        results[query_name] = {
            "avg_time": sum(times) / len(times),
            "min_time": min(times),
            "max_time": max(times)
        }
    
    return results

# Clear cache for fair benchmarking
prod_engine.clear_cache()

benchmark_results = benchmark_queries(prod_engine, test_queries, iterations=3)

print("Benchmark Results (3 iterations each):")
for query_name, metrics in benchmark_results.items():
    print(f"  {query_name}:")
    print(f"    Avg: {metrics['avg_time']:.4f}s")
    print(f"    Range: [{metrics['min_time']:.4f}s - {metrics['max_time']:.4f}s]")

print("\n✅ Production GraphQL Core integration and performance optimization completed")